# Retrieval as Bayesian conditioning — interactive companion

Companion to [Post 4a: Retrieval as Bayesian Conditioning](../posts/04a-rag-as-bayesian-conditioning.qmd).

RAG is Bayesian inference: parametric memory is a prior, retrieved
documents are evidence, the answer is a posterior. This notebook lets you
watch a wrong prior get corrected by evidence, see when retrieval helps vs
hurts, and build the confidence-gated policy that beats both always- and
never-retrieve.

**You'll do (~20 minutes):**
1. Watch prior x likelihood = posterior on one query.
2. See retrieval help most when the model knows least.
3. Find where retrieval *hurts* a knowledgeable model.
4. Build confidence-gated retrieval and beat both extremes.

## 0. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (9, 4.5)
plt.rcParams["figure.dpi"] = 110

from nano_agents.retrieval import (
    RetrievalQA, parametric_prior, posterior, answer_accuracy,
)

rng = np.random.default_rng(0)

## 1. The Bayesian update on one query

The model has a prior over answers from parametric memory. Retrieved
documents each vote for the answer they support. The posterior is
prior x likelihood, normalized.

In [ ]:
env = RetrievalQA(n_queries=30, n_answers=5, n_relevant=3,
                  n_distractors=150, relevance=0.8, embed_dim=16, seed=1)
q = 0
correct = int(env.correct_answers[q])
n = env.n_answers

# A confidently-WRONG prior (the model misremembers).
wrong = (correct + 2) % n
prior = np.full(n, 0.05); prior[wrong] = 0.6; prior[correct] = 0.15
prior = prior / prior.sum()

docs = env.retrieve(q, k=3)
post = posterior(env, prior, docs, evidence_strength=2.0)

print(f"correct answer: A{correct}")
print(f"prior:     {np.round(prior, 2)}  -> argmax A{int(prior.argmax())} (WRONG)")
print(f"posterior: {np.round(post, 2)}  -> argmax A{int(post.argmax())}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, vals, title, c in [(axes[0], prior, "Prior (misremembers)", "#dd8452"),
                            (axes[1], post, "Posterior (corrected)", "#55a467")]:
    bars = ax.bar(range(n), vals, color=c, edgecolor="white")
    bars[correct].set_edgecolor("#222"); bars[correct].set_linewidth(2.5)
    ax.set_xticks(range(n)); ax.set_xticklabels([f"A{i}" for i in range(n)])
    ax.set_title(title); ax.set_ylim(0, 1)
plt.tight_layout(); plt.show()

The black-outlined bar is the correct answer. Retrieval flips the model
from confidently wrong to confidently right.

### Try this
- Set `evidence_strength=0.5`. Weak evidence may not overcome the wrong prior.
- Set the prior to be confidently *correct* and watch what 3 distractor-heavy
  retrievals do at high evidence strength.

## 2. Retrieval helps most when the model knows least

Sweep parametric knowledge; compare closed-book (prior only) to open-book
(prior + retrieval).

In [ ]:
env = RetrievalQA(n_queries=80, n_answers=5, n_relevant=3,
                  n_distractors=250, relevance=0.75, embed_dim=16, seed=0)
knowledge_levels = np.linspace(0, 1, 11)
closed, openbook = [], []
for kn in knowledge_levels:
    closed.append(answer_accuracy(env, knowledge=kn, k=0, n_trials=15,
                                   use_retrieval=False, seed=0))
    openbook.append(answer_accuracy(env, knowledge=kn, k=3, n_trials=15,
                                     evidence_strength=2.0, seed=0))

plt.plot(knowledge_levels, closed, "o-", color="#dd8452", lw=2,
         label="closed-book")
plt.plot(knowledge_levels, openbook, "s-", color="#3a7ebf", lw=2,
         label="open-book (RAG)")
plt.fill_between(knowledge_levels, closed, openbook, color="#3a7ebf", alpha=0.1)
plt.xlabel("parametric knowledge"); plt.ylabel("accuracy")
plt.title("Retrieval helps most when the model knows least")
plt.legend(); plt.grid(alpha=0.3); plt.ylim(0, 1.05); plt.show()

The shaded gap is the value of retrieval — biggest when knowledge is low.

### Try this
- Lower `relevance` to 0.4 (worse retriever). Does open-book still dominate?
- The open-book curve is flat near 1.0 because relevance=0.75 gives perfect
  precision@3. Drop it and watch the curve sag.

## 3. When retrieval HURTS

A knowledgeable model with a poor retriever can be dragged below its
closed-book accuracy by misleading evidence.

In [ ]:
relevances = np.linspace(0.15, 0.9, 16)
knowledge = 0.9  # the model already knows most answers

env0 = RetrievalQA(n_queries=80, n_answers=5, n_relevant=3,
                   n_distractors=250, relevance=0.7, embed_dim=16, seed=0)
closed = answer_accuracy(env0, knowledge=knowledge, k=0, n_trials=20,
                          use_retrieval=False, seed=0)

openbook = []
for rel in relevances:
    env = RetrievalQA(n_queries=80, n_answers=5, n_relevant=3,
                      n_distractors=250, relevance=float(rel), embed_dim=16, seed=0)
    openbook.append(answer_accuracy(env, knowledge=knowledge, k=3,
                                     evidence_strength=2.0, n_trials=20, seed=0))

plt.plot(relevances, openbook, "o-", color="#3a7ebf", lw=2, label="open-book")
plt.axhline(closed, color="#dd8452", ls="--", lw=2,
            label=f"closed-book = {closed:.2f}")
plt.xlabel("retriever quality"); plt.ylabel("accuracy")
plt.title("Below a quality threshold, retrieval HURTS a knowledgeable model")
plt.legend(); plt.grid(alpha=0.3); plt.ylim(0, 1.05); plt.show()

Where the blue curve dips below the orange line, retrieval is net harmful —
bad evidence overrides correct priors.

### Try this
- Set `knowledge=0.3`. The crossover shifts left: a less knowledgeable model
  benefits from retrieval even when it's mediocre (it has less to lose).

## 4. Confidence-gated retrieval

The fix: retrieve only when the prior is uncertain. This needs the model to
*know when it doesn't know* — i.e. to be calibrated.

In [ ]:
env = RetrievalQA(n_queries=80, n_answers=5, n_relevant=2,
                  n_distractors=400, relevance=0.45, embed_dim=16, seed=0)
knowledge, k, strength, n_trials = 0.6, 6, 2.0, 25

def gated(threshold):
    r = np.random.default_rng(0); correct = total = 0
    for _ in range(n_trials):
        for q in range(env.n_queries):
            prior = parametric_prior(env, q, knowledge, r)
            if prior.max() < threshold:
                post = posterior(env, prior, env.retrieve(q, k), strength)
            else:
                post = prior
            correct += int(np.argmax(post) == env.correct_answers[q]); total += 1
    return correct / total

never = answer_accuracy(env, knowledge, 0, strength, n_trials,
                         use_retrieval=False, seed=0)
always = answer_accuracy(env, knowledge, k, strength, n_trials,
                          use_retrieval=True, seed=0)
thresholds = np.linspace(0.2, 1.0, 17)
gated_acc = [gated(t) for t in thresholds]

plt.plot(thresholds, gated_acc, "o-", color="#55a467", lw=2, label="gated")
plt.axhline(never, color="#dd8452", ls="--", lw=2, label=f"never = {never:.2f}")
plt.axhline(always, color="#c44e52", ls="--", lw=2, label=f"always = {always:.2f}")
plt.xlabel("confidence threshold"); plt.ylabel("accuracy")
plt.title("Confidence-gated retrieval beats both extremes")
plt.legend(); plt.grid(alpha=0.3); plt.show()

print(f"never={never:.2f}, always={always:.2f}, best gated={max(gated_acc):.2f}")

The gated policy beats both always- and never-retrieve — but only because
the model's confidence is a trustworthy signal. That's calibration, the
subject of Post 4c.

### Try this
- Make the prior unreliable: a model that's confidently wrong will skip
  retrieval exactly when it needs it. (Hint: this is what *miscalibration*
  does, and why gating fails without calibration.)

## What's next

You've seen RAG as Bayesian conditioning:

- **Prior x likelihood = posterior** — parametric memory updated by evidence.
- **Retrieval helps most when the model knows least** — the value is the gap.
- **Retrieval can hurt** — bad evidence overrides good priors.
- **Confidence-gating** — retrieve only when uncertain, if calibrated.

The next post — [Post 4b: Self-reflection as Monte Carlo](../posts/04b-self-reflection.qmd) —
turns from external evidence (retrieval) to internal evidence: the model
critiquing and revising its own output. We'll see when reflection genuinely
improves answers and when it just amplifies the model's own biases.